#7 - Gold Intraday - Atualização Constante

---

Importando todas as bibliotecas que serão utilizadas durante o notebook, em seguida definimos os caminhos das tabelas que serão utilizadas durante a execução.

#.
### Variáveis e suas utilizações
micro_silver_path = define o caminho de leitura da tabela silver do micro batch

gold_intraday_path = define o caminho onde será criada a tabela gold intraday, utilizada para alimentar o gráfico de variação minuto a minuto

---

In [0]:
import pyspark.sql.functions as sf

micro_silver_path = "workspace.stocks.micro_batch_silver"
gold_intraday_path = "workspace.stocks.gold_intraday"

---

Verificamos o último timestamp salvo na tabela gold intraday para garantir que somente registros novos sejam processados a cada execução, evitando duplicidade de dados. Caso a tabela ainda não exista, retornamos None e processamos tudo.

---

In [0]:
try:
      ultimo_ts = (
            spark.read.table(gold_intraday_path)
            .agg(sf.max("event_time"))
            .collect()[0][0]
      )
      print(f"-Ultimo TimeStamp: {ultimo_ts}-")
except:
      ultimo_ts = None

---

Lemos a tabela silver do micro batch e padronizamos todos os valores numéricos aplicando round de duas casas decimais nas colunas de preço e variação, mantendo a precisão adequada para exibição no gráfico intraday.

---

In [0]:
df = spark.read.table(micro_silver_path)

df_gold_intraday = (df
      .withColumn("open", sf.round(sf.col("open"),2))
      .withColumn("high", sf.round(sf.col("high"),2))
      .withColumn("low", sf.round(sf.col("low"),2))
      .withColumn("close", sf.round(sf.col("close"),2))
      .withColumn("variacao_real", sf.round(sf.col("variacao_real"),2))
      .withColumn("variacao_percent", sf.round(sf.col("variacao_percent"),2))
)


---

Filtramos apenas os registros mais recentes que o último timestamp salvo,
garantindo que somente candles novos sejam adicionados à tabela gold intraday.

Salvamos em append com particionamento por ticker para otimizar as consultas do gráfico.

---

In [0]:
if ultimo_ts is not None:
      df_gold_intraday = df_gold_intraday.filter(sf.col("event_time") > ultimo_ts)

(
   df_gold_intraday.write
    .format("delta")
    .mode("append")
    .partitionBy("ticker")
    .saveAsTable(gold_intraday_path)
)

---

Verificamos se novos registros foram gravados e exibimos uma amostra dos dados do primeiro ticker ordenados por event_time para confirmar a correta atualização da tabela.

---

In [0]:
if df_gold_intraday.count() > 0:
      print("-Dados gravados-")
      (df_gold_intraday
      .filter(sf.col("ticker") == df_gold_intraday.first()["ticker"])
      .orderBy(sf.col("event_time"))
      .display())
else:
    print("-Sem valores novos-")